In [ ]:
#librerias
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os 
pio.renderers.default = "notebook" 

#configuracion de carpetas 
CARPETA_SALIDA = "Graficos_Informe"

#Creamos la carpeta automáticamente
if not os.path.exists(CARPETA_SALIDA):
    os.makedirs(CARPETA_SALIDA)
    print(f"Carpeta creada: {CARPETA_SALIDA}")
else:
    print(f"La carpeta '{CARPETA_SALIDA}' ya existe. Los gráficos se guardarán ahí.")

#paleta
COLORES = {
    'Verde_Fuerte': '#74b404', 
    'Verde_Claro':  '#cdfc7d', 
    'Rojo_Corp':    '#aa044c', 
    'Morado_Os':    '#876784',
    'Morado_Cl':    '#b09eae'
}

#Creamos funcion para guardar aqui los graficos
def guardar_grafico(fig, nombre_archivo):

    ruta_png = os.path.join(CARPETA_SALIDA, f"{nombre_archivo}.png")
    fig.write_image(ruta_png, width=1200, height=800, scale=2)
    
    print(f"Gráfico guardado: {ruta_png}")

#Carga de datos
def cargar_y_preparar_datos(ruta_archivo):
    print(f"Cargando datos desde: {ruta_archivo}")
    try:
        df = pd.read_excel(ruta_archivo)
        
        df_viv = df[df['Proposito'].astype(str).str.contains('Vivienda', case=False, na=False)].copy()
        df_viv['Impago_Label'] = df_viv['Impago'].map({0: 'Pagado', 1: 'Impago'})
        
        if 'Posesion_Hipoteca' in df_viv.columns:
            df_viv['Tiene_Hipoteca'] = df_viv['Posesion_Hipoteca'].map({0: 'No tiene', 1: 'Sí tiene'})
            
        col_fiador = 'Fiador' if 'Fiador' in df_viv.columns else 'Cofirmante'
        df_viv['Fiador_Label'] = df_viv[col_fiador].map({0: 'Sin Fiador', 1: 'Con Fiador'})
            
        return df_viv
    except Exception as e:
        print(f"Error: {e}")
        return None

#Cargamos
ruta_real = os.path.join('..', 'Datos', 'Limpios', 'información_préstamos_limpio.xlsx')
df_viv = cargar_y_preparar_datos(ruta_real)

In [ ]:
#GRAFICO 1
fig1 = px.violin(
    df_viv, 
    y="Ratio_Deuda_Ingresos", 
    x="Impago_Label", 
    color="Impago_Label",
    box=True,               
    points="all",            
    hover_data=['Ingresos', 'Scoring_Crediticio'], 

    color_discrete_map={
        'Pagado': COLORES['Verde_Fuerte'], 
        'Impago': COLORES['Rojo_Corp']
    }, 
    title="<b>Distribución del Endeudamiento (Ratio Deuda/Ingresos)</b><br><sup>Préstamos de Vivienda: Clientes al Corriente vs. En Mora</sup>"
)


fig1.update_layout(
    xaxis_title="Estado del Préstamo",
    yaxis_title="Ratio Deuda / Ingresos (%)",
    legend_title="Estado",
    template="plotly_white",
    font=dict(size=12),
    title_font_size=16
)

fig1.show()
guardar_grafico(fig1, "1_Perfil_Riesgo")

In [ ]:
#GRAFICO 2
fig2 = px.scatter(
    df_viv, 
    x="Ingresos", 
    y="Monto_Inicial",
    color="Impago_Label", 
    facet_col="Impago_Label",    
    size="Ratio_Deuda_Ingresos", 
    size_max=10, 
    opacity=0.3,                 
    hover_data=['Scoring_Crediticio'],
    
    color_discrete_map={
        'Pagado': COLORES['Verde_Fuerte'], 
        'Impago': COLORES['Rojo_Corp']
    },
    title="<b>Mapa de Riesgo Comparativo: Ingresos vs. Monto Inicial</b><br><sup>Segmentación de Vivienda: Análisis de densidad y concentración de impagos</sup>"
)


fig2.update_layout(
    template="plotly_white",
    xaxis_title="Ingresos Anuales",
    yaxis_title="Monto del Préstamo",
    legend_title="Estado"
)


fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig2.show()
guardar_grafico(fig2, "2_Ingresos_monto")


In [ ]:
#GRAFICO 3
variables = ['Estado_Civil', 'Tipo_Jornada_Laboral', 'Estudios']
prefijos = ['3a', '3b', '3c'] # Para nombrar los archivos ordenadamente

for var, pre in zip(variables, prefijos):
    df_agrupado = df_viv.groupby([var, 'Impago_Label']).size().reset_index(name='Cantidad')
    df_agrupado['Porcentaje'] = df_agrupado.groupby(var)['Cantidad'].transform(lambda x: 100 * x / x.sum())
    
    fig3 = px.bar(
        df_agrupado,
        x=var,
        y='Porcentaje',
        color='Impago_Label',
        text=df_agrupado['Porcentaje'].apply(lambda x: '{0:1.2f}%'.format(x)),
        color_discrete_map={
            'Pagado': COLORES['Verde_Fuerte'], 
            'Impago': COLORES['Rojo_Corp']
        },
        title=f"<b>Riesgo de Impago por {var.replace('_', ' ')}</b>"
    )
    
    fig3.update_layout(
        yaxis_title="Porcentaje (%)",
        xaxis_title=var.replace('_', ' '),
        template="plotly_white",
        uniformtext_minsize=8, 
        uniformtext_mode='hide'
    )
    
    fig3.show()
    guardar_grafico(fig3, f"{pre}_{var}")

In [ ]:
# GRÁFICO 4:
col_fiador = 'Fiador' 

df_viv['Fiador_Label'] = df_viv[col_fiador].map({0: 'Sin Fiador', 1: 'Con Fiador'})

fig4 = px.histogram(
    df_viv,
    x="Ratio_Interes",  
    color="Fiador_Label",
    marginal="box",     
    nbins=30, 
    opacity=0.7, 
    barmode="overlay", 
    color_discrete_map={
        'Sin Fiador': COLORES['Rojo_Corp'], 
        'Con Fiador': COLORES['Verde_Fuerte']
    },
    title="<b>Distribución de Tipos de Interés en Vivienda</b><br><sup>Impacto del Fiador en el coste del préstamo</sup>"
)


fig4.update_layout(
    xaxis_title="Tasa de Interés (%)",
    yaxis_title="Frecuencia (Nº de Préstamos)",
    template="plotly_white",
    legend_title="¿Tiene Fiador?",
    font=dict(size=12)
)

fig4.show()
guardar_grafico(fig4, "4_Distribucion_Interes")

In [ ]:
# GRÁFICO 7
bins_edad = [18, 25, 35, 45, 55, 65, 120]
labels_edad = ['18-25', '26-35', '36-45', '46-55', '56-65', '+65']

df_viv['Rango_Edad'] = pd.cut(df_viv['Edad'], bins=bins_edad, labels=labels_edad)

fig7 = px.box(
    df_viv,
    x='Rango_Edad',
    y='Meses_Empleo',
    color='Impago_Label',

    category_orders={"Rango_Edad": labels_edad}, 
    title='<b>Perfil de Estabilidad: Antigüedad Laboral por Edad</b><br><sup>Comparativa de meses en el empleo actual</sup>',
    labels={'Rango_Edad': 'Rango de Edad', 'Meses_Empleo': 'Meses de Antigüedad'},
    color_discrete_map={
        'Pagado': COLORES['Verde_Fuerte'], 
        'Impago': COLORES['Rojo_Corp']
    },
    template='plotly_white'
)

fig7.update_layout(
    xaxis_title="Grupos de Edad (Ordenados)",
    yaxis_title="Meses en el Empleo Actual",
    legend_title="Estado",
    boxmode='group' 
)

fig7.show()
guardar_grafico(fig7, "7_Estabilidad_Laboral")